In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import base64
import os
import json
import os
from io import BytesIO
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText


In [2]:
# Install required libraries for Qwen2.5-VL-3B-Instruct
!pip install -q torchvision accelerate transformers>=4.48.0 qwen-vl-utils

QWEN25_MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"

# Initialize processor and model
processor_qwen25 = AutoProcessor.from_pretrained(QWEN25_MODEL_NAME)

model_qwen25 = AutoModelForImageTextToText.from_pretrained(
    QWEN25_MODEL_NAME,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

model_qwen25.eval()

print("Qwen2.5-VL-3B-Instruct loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Qwen2.5-VL-3B-Instruct loaded successfully!


In [3]:
def query_from_path(image_path):
    # Helper to look up the correct query based on the image_path file name
    target_name = "images/" + os.path.basename(image_path)
    for record in annotations_list:
        if record["image_path"] == target_name:
            return record["question"]
    raise ValueError(f"No annotation record found for image path: {image_path}")

In [4]:
def get_base64_from_rgb(rgb_image):
    # Helper to convert in-memory numpy array (RGB) to base64 string
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

In [5]:
def evaluate_original(image_path):
  with open(image_path, "rb") as f:
    base64_image = base64.b64encode(f.read()).decode("utf-8")
  query_text = query_from_path(image_path)
  # Predict
  result = evaluate(query_text,base64_image)
  return result

In [6]:
def evaluate(query_text, b64_str):
    # Decode the base64 string to a PIL Image
    image_bytes = base64.b64decode(b64_str)
    image = Image.open(BytesIO(image_bytes)).convert("RGB")

    question = query_text + "\nAnswer only Yes or No."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor_qwen25.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor_qwen25(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model_qwen25.device)

    with torch.inference_mode():
        output_ids = model_qwen25.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False
        )

    generated_ids = output_ids[:, len(inputs["input_ids"][0]):]

    answer = processor_qwen25.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return answer.strip()

In [9]:
import os
import json
import cv2
import numpy as np
import base64
import pandas as pd
from tqdm.notebook import tqdm

ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/pope-dataset/annotations.json"

# 1. Load all annotations
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Filter to keep only records where image_name starts with '/images/'
filtered_annotations = [
    record for record in annotations_list
    if record.get("image_name", "").startswith("/images/")
]

# 2. Create a blank grey image in memory (e.g., 384x384)
blank_grey_img = np.full((384, 384, 3), 128, dtype=np.uint8)


# Encode the single blank image to base64 once
base64_image = get_base64_from_rgb(blank_grey_img)

results = []

print(f"Processing single blank image through {len(filtered_annotations)} queries starting with /images/...")
for record in tqdm(filtered_annotations):
    image_id = record.get("image_id")
    ground_truth = record.get("label", "Unknown")
    query_text = record.get("query", record.get("question", ""))

    if not query_text:
        continue

    # Evaluate the single blank image under the original condition
    try:
        orig_val = evaluate(query_text, base64_image)
    except Exception as e:
        orig_val = f"Error: {str(e)}"

    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val
    })

# Create final DataFrame
df_results = pd.DataFrame(results)
display(df_results.head(10))

# Save to guessing.csv
df_results.to_csv("guessing1.csv", index=False)
print("Saved results to guessing1.csv")

Processing single blank image through 500 queries starting with /images/...


  0%|          | 0/500 [00:00<?, ?it/s]

,Image_Id,Ground_truth,Original
0,46207,No,No
1,489255,No,No.
2,88567,No,No.
3,311612,No,No.
4,48900,No,No.
5,452250,No,No.
6,429312,No,No.
7,433554,No,No
8,253177,No,No.
9,528582,No,No.


Saved results to guessing1.csv


In [14]:
ANNOTATIONS_PATH = "/kaggle/input/datasets/ramakanthjogada/true-positive-only/repope_yes_500_unique/annotations.json"

# 1. Load all annotations
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_list = json.load(f)

# Filter to keep only records where image_name starts with '/images/'
filtered_annotations = [
    record for record in annotations_list
    if record.get("image_path", "").startswith("images/")
]

# 2. Create a blank grey image in memory (e.g., 384x384)
blank_grey_img = np.full((384, 384, 3), 128, dtype=np.uint8)

# Helper to convert in-memory numpy array to base64 string
def get_base64_from_rgb(rgb_image):
    bgr_img = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    _, buffer = cv2.imencode('.jpg', bgr_img)
    return base64.b64encode(buffer).decode('utf-8')

# Encode the single blank image to base64 once
base64_image = get_base64_from_rgb(blank_grey_img)

results = []

print(f"Processing single blank image through {len(filtered_annotations)} queries starting with images/...")
for record in tqdm(filtered_annotations):
    image_id = record.get("image_id")
    ground_truth = record.get("label", "Unknown")
    query_text = record.get("query", record.get("question", ""))

    if not query_text:
        continue

    # Evaluate the single blank image under the original condition
    try:
        orig_val = evaluate(query_text, base64_image)
    except Exception as e:
        orig_val = f"Error: {str(e)}"

    results.append({
        "Image_Id": image_id,
        "Ground_truth": ground_truth,
        "Original": orig_val
    })

# Create final DataFrame
df_results = pd.DataFrame(results)
display(df_results.head(10))

# Save to guessing.csv
df_results.to_csv("guessing2.csv", index=False)
print("Saved results to guessing2.csv")

Processing single blank image through 491 queries starting with /images/...


  0%|          | 0/491 [00:00<?, ?it/s]

,Image_Id,Ground_truth,Original
0,000000310196,yes,No.
1,000000210789,yes,No.
2,000000429109,yes,No.
3,000000211674,yes,No.
4,000000458338,yes,No.
5,000000283412,yes,No.
6,000000265719,yes,No.
7,000000461331,yes,No.
8,000000544456,yes,No.
9,000000017708,yes,No.


Saved results to guessing2.csv
